In [2]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware, before_agent, HumanInTheLoopMiddleware, wrap_model_call, ModelRequest, ModelResponse, dynamic_prompt
from langchain.messages import HumanMessage, AIMessage, ToolMessage, RemoveMessage
from langgraph.runtime import Runtime
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SQLDatabase

from tavily import TavilyClient

from dataclasses import dataclass
from dotenv import load_dotenv
from pprint import pprint
from typing import Dict, Any, Callable
import os

In [3]:
load_dotenv()

True

In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile" ,
    groq_api_key=os.getenv("GROQ_API_KEY") ,
    temperature=0 ,
)

In [5]:
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash", 
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

## Summarize messages

In [8]:
summarize_agent = create_agent(
    model = llm,
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [9]:
response = summarize_agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

In [6]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to gather information about Lunapolis, the capital of the moon, including its weather and details about its inhabitants, specifically the cheese miners.

## SUMMARY
The capital of the moon is Lunapolis, with extreme weather conditions (high of 120C and low of -100C). There are 100,000 cheese miners living in Lunapolis, and they are likely to go on strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
Investigate the reasons behind the cheese miners' dissatisfaction with the new president and the potential impact of their strike on Lunapolis.


# Trim/delete messages

In [14]:
@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [15]:
trim_agent = create_agent(
    model = llm,
    checkpointer = InMemorySaver(),
    middleware = [trim_messages]
)

In [16]:
response = trim_agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='7e2d5e71-dd70-4e2f-bb6e-ea0308c4815c'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='04e514ce-e11c-43ec-9aeb-b7ab3e3a46dc', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='28dab8f4-fd7d-402b-871a-ef730a1e0092'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='802a2c82-8c54-49be-966e-18c1202d9dd0', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='36ab8c1e-9a40-4c8c-9339-82876564e740'),
              AIMessage(content="If the device has been exposed to extreme temperatures (very h

In [10]:
print(response["messages"][-1].content)

If the device has been exposed to extreme temperatures (very hot or cold), it may not turn on. Try to check if the device is at a normal room temperature (around 60-80°F or 15-27°C). If it's been in a hot or cold environment, let it sit at room temperature for a while before trying to turn it on again.

Also, try the following basic troubleshooting steps:

1. Check the power cord and ensure it's properly connected to both the device and the power outlet.
2. Try pressing the power button for a longer duration (around 10-15 seconds) to see if it turns on.
3. If the device has a removable battery, try taking it out and putting it back in.

If none of these steps work, it's possible that there's a more serious issue with the device, and you may need to contact the manufacturer or a professional for further assistance.


# HITL(Human In The Loop)

In [19]:
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [21]:
class EmailState(AgentState):
    email: str

hitl_agent = create_agent(
    model=llm,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [22]:
config = {"configurable": {"thread_id": "1"}}

response = hitl_agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [23]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Re: '
                                                                          'Email '
                                                                          'Response'},
                                                         'description': 'Tool '
                                                                        'execution '
                                                                        'requires '
                                                                        'approval\n'
                                                                        '\n'
                                                                        'Tool: '
                                                                        'send_email\n'
                                                                        'Args: '
                                                                        "{'body': "
     

In [24]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Re: Email Response'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Re: Email Response'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='1089e75c93d2704ee43fd99244ec3cf5')]


In [25]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Re: Email Response


In [26]:
response = hitl_agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Re: '
                                                                          'Email '
                                                                          'Response. '
                                                                          'Sure '
                                                                          'John, '
                                                                          'what '
                                                                          'time '
                                                                          'works '
                                                                          'for '
                                                                          'you?'},
                                                         'description': 'Tool '
                                                                        'execution '
        

In [27]:
response = hitl_agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Re: '
                                                                          'Email '
                                                                          'Response. '
                                                                          'Sure '
                                                                          'John, '
                                                                          'what '
                                                                          'time '
                                                                          'works '
                                                                          'for '
                                                                          'you? '
                                                                          '- '
                                                                          'Your '
             

In [28]:
response = hitl_agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': "You're "
                                                                          'fired! '
                                                                          '- '
                                                                          'Your '
                                                                          'merciful '
                                                                          'leader, '
                                                                          'Seán'},
                                                         'description': 'Tool '
                                                                        'execution '
                                                                        'requires '
                                                                        'approval\n'
                                                                        '\n'
      

# Dynamic Agent

In [36]:
@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = gemini_llm
    else:
        # Short conversation - use efficient model
        model = llm

    request = request.override(model=model)  

    return handler(request)

In [37]:
dynamic_agent = create_agent(
    model=llm,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [39]:
response = dynamic_agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)
print(response["messages"][-1].response_metadata["model_name"])

I actually watered the office plant about an hour ago. I checked the soil and it was feeling a bit dry, so I gave it a good soaking. I also rotated the pot a bit to make sure it's getting even sunlight. It's looking pretty healthy, if I do say so myself. Would you like me to check on anything else around the office?
llama-3.3-70b-versatile


In [41]:
response = dynamic_agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)
print(response["messages"][-1].response_metadata["model_name"])

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 30.507442186s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url

# Dynamic Prompt

In [43]:
@dataclass
class LanguageContext:
    user_language: str = "English"

In [45]:
@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [46]:
dynamic_prompt_agent = create_agent(
    model=llm,
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [49]:
response = dynamic_prompt_agent.invoke(
    {"message": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Bangla")
)

print(response["messages"][-1].content)

আমি আপনাকে সাহায্য করতে প্রস্তুত। আপনার কী প্রয়োজন?


# Dynamic Tool

In [61]:
tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

In [62]:
@dataclass
class UserRole:
    user_role: str = "external"

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [71]:
@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [72]:
dynamic_tool_agent = create_agent(
    model=llm,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [73]:
response = dynamic_tool_agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=web_search{"query": "number of artists in database"}</function>'}}

# Email Agent

In [6]:
@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

class AuthenticatedState(AgentState):
    authenticated: bool

In [7]:
@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

@tool
def request_human_feedback(question: str) -> str:
    """
    Call this tool when you need human approval, 
    missing information, or a final review of a complex task.
    """
    # This return value will only be seen if you 'Approve' without editing.
    return f"Human has reviewed the question: {question}"

In [8]:
@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [9]:
email_agent = create_agent(
    model=llm,
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
                # "request_human_feedback": True;
            })
        ]
    )


In [83]:
config = {"configurable": {"thread_id": "1"}}

response = email_agent.invoke(
    {"messages": [HumanMessage(content="draft 1")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

I'm happy to help you with drafting, but I don't see any specific details about what you would like to draft. Could you please provide more context or information about what you need help with?


In [12]:
config = {"configurable": {"thread_id": "1"}}

while True :
    user_input = input()

    if "bye" in user_input :
        break

    response = email_agent.invoke(
        {'messaage': [HumanMessage(content=user_input)]},
        context=EmailContext(),
        config=config
    )

    print(response['messages'][-1].content)

    if "__interrupt__" in response:
        interrupt = response["__interrupt__"][0] # Get the first pending tool call
        tool_call = interrupt["action_request"]
        
        print(f"AI wants to call: {tool_call['name']} with {tool_call['args']}")
        
        # --- USER INPUT LOGIC ---
        choice = input("Enter 'a' for Approve, 'r' for Reject, 'e' for Edit: ").lower()

        if choice == 'a':
            # APPROVE
            resumed_response = hitl_agent.invoke(Command(resume="approve"), config=config)
        elif choice == 'r':
            # REJECT
            resumed_response = hitl_agent.invoke(Command(resume="reject"), config=config)
        elif choice == 'e':
            # EDIT: Let's say you want to change one argument
            new_args = tool_call['args'].copy()
            new_args['email_body'] = "Modified by Human: " + new_args.get('email_body', '')
            
            # Pass the edited tool call back
            resumed_response = hitl_agent.invoke(
                Command(resume={"action_request": {**tool_call, "args": new_args}}), 
                config=config
            )

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expe

I am not able to authenticate the user with the given credentials. If you could provide the correct email and password, I would be happy to try again.



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(


I am not able to authenticate the user with the given credentials. If you could provide the correct email and password, I would be happy to try again.



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(


I am not able to authenticate the user with the given credentials. If you could provide the correct email and password, I would be happy to try again.
